<a href="https://colab.research.google.com/github/Chimatanagagopal/RAG/blob/main/simplepdf_QA_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 10.7 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()

Saving PYTHON PROGRAMMING NOTES.pdf to PYTHON PROGRAMMING NOTES.pdf


In [3]:
from pypdf import PdfReader

pdf_name = list(uploaded.keys())[0]

reader = PdfReader(pdf_name)

# Extract all text
full_text = ""

for page in reader.pages:
    text = page.extract_text()

    if text:
        full_text += text + "\n"

print("Total characters:", len(full_text))


Total characters: 139362


In [4]:
chunk_size = 1000
overlap = 200

chunks = []

start = 0

while start < len(full_text):
    end = start + chunk_size

    chunk = full_text[start:end]

    chunks.append(chunk)

    start += chunk_size - overlap

print("Number of chunks:", len(chunks))

Number of chunks: 175


In [5]:
!pip install -q sentence-transformers

In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
embeddings = model.encode(chunks)

print(type(embeddings))
print("Number of embeddings:", len(embeddings))
print("Embedding shape:", embeddings.shape)

<class 'numpy.ndarray'>
Number of embeddings: 175
Embedding shape: (175, 384)


In [8]:
# print("TEXT:")
# print(chunks[0])

# print("\nVECTOR:")
# print(embeddings[0])

In [9]:
print(embeddings.shape)

(175, 384)


In [10]:
from sklearn.metrics.pairwise import cosine_similarity
!pip install -q scikit-learn

In [11]:
query = "What are fruitful functions in Python?"

In [12]:
query_embedding = model.encode([query])

print(query_embedding.shape)

(1, 384)


In [13]:
similarities = cosine_similarity(
    query_embedding,
    embeddings
)

print(similarities.shape)

(1, 175)


In [14]:
import numpy as np

scores = similarities[0]

top_indices = np.argsort(scores)[::-1][:5]

for index in top_indices:
    print("\nScore:", scores[index])



Score: 0.7055299

Score: 0.6116135

Score: 0.61126024

Score: 0.59824514

Score: 0.5969907


In [15]:
for rank, index in enumerate(top_indices, start=1):
    print(f"\n===== RANK {rank} =====")
    print("Score:", scores[index])
    print("Chunk:")
    print(chunks[index][:1000])


===== RANK 1 =====
Score: 0.7055299
Chunk:
                 MRCET 
54 
 
>>> 
 
Similarily we can also write,  
 
def f(arg): pass    # a function that does nothing (yet) 
 
class C: pass       # a class with no methods (yet) 
  
PYTHON PROGRAMMING                                      III YEAR/II SEM                  MRCET 
55 
 
UNIT – III 
FUNCTIONS, ARRAYS  
Fruitful functions: return values, parameters, local and global scope, function composition, 
recursion; Strings: string slices, immutability, string functions and methods, string module; 
Python arrays, Access the Elements of an Array, array methods. 
Functions, Arrays: 
Fruitful functions: 
We write functions that return values, which we will call  fruitful functions. We have seen 
the return statement before, but in a fruitful function the  return statement includes a  return 
value. This statement means: "Return immediately from this function and use the following 
expression as a return value."  
(or) 
Any function that re

In [16]:
import torch

print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

GPU available: True
GPU: Tesla T4


In [17]:
!pip install -q faiss-cpu
import faiss

print("FAISS version:", faiss.__version__)
print("Number of GPUs detected by FAISS:", faiss.get_num_gpus())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 71.8 MB/s eta 0:00:00
FAISS version: 1.15.1
Number of GPUs detected by FAISS: 0


In [18]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Index type:", type(index))
print("Vectors stored:", index.ntotal)

Index type: <class 'faiss.swigfaiss.IndexFlatL2'>
Vectors stored: 175


In [19]:
query = "What are fruitful functions in Python?"

query_embedding = model.encode([query])

D, I = index.search(query_embedding, 5)

print("Distances:")
print(D)

print("Chunk indices:")
print(I)

Distances:
[[0.5889403  0.7767731  0.77747965 0.80350995 0.80601865]]
Chunk indices:
[[70  1 34 80 31]]


In [20]:
for rank, (distance, chunk_index) in enumerate(zip(D[0], I[0]), start=1):
    print(f"\n===== RANK {rank} =====")
    print("Distance:", distance)
    print("Chunk index:", chunk_index)
    print("Chunk:")
    print(chunks[chunk_index])


===== RANK 1 =====
Distance: 0.5889403
Chunk index: 70
Chunk:
                 MRCET 
54 
 
>>> 
 
Similarily we can also write,  
 
def f(arg): pass    # a function that does nothing (yet) 
 
class C: pass       # a class with no methods (yet) 
  
PYTHON PROGRAMMING                                      III YEAR/II SEM                  MRCET 
55 
 
UNIT – III 
FUNCTIONS, ARRAYS  
Fruitful functions: return values, parameters, local and global scope, function composition, 
recursion; Strings: string slices, immutability, string functions and methods, string module; 
Python arrays, Access the Elements of an Array, array methods. 
Functions, Arrays: 
Fruitful functions: 
We write functions that return values, which we will call  fruitful functions. We have seen 
the return statement before, but in a fruitful function the  return statement includes a  return 
value. This statement means: "Return immediately from this function and use the following 
expression as a return value."  
(or) 
A

In [21]:
!pip install -q transformers accelerate

In [22]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base").to(device)

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [24]:
context = "\n\n".join([chunks[i] for i in I[0]])

prompt = f"""You are a helpful teacher. Explain in detail based on the context below.
Cover all points clearly with examples.

Context:
{context}

Question: {query}

Detailed Explanation:"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(device)

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
    repetition_penalty=2.0
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(answer)

PYTHON PROGRAMMING OBJECTIVES:  To read and write simple Python programs.  To develop Python programs with conditionals and loops.  To define Python functions and call them.  To use Python data structures –- lists, tuples, dictionaries.  To do input/output with files in Python.


In [25]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")
model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    device_map="auto",
    torch_dtype=torch.float16
)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [26]:
context = "\n\n".join([chunks[i] for i in I[0]])

prompt = f"""<s>[INST] You are a helpful Python teacher.
Using the context below, explain in detail with examples like you are teaching a student.

Context:
{context}

Question: {query} [/INST]"""

inputs = tokenizer(prompt, return_tensors="pt").to(device)

outputs = model.generate(
    **inputs,
    max_new_tokens=1024,
    do_sample=True,
    temperature=0.7,
    repetition_penalty=1.3
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(answer)

[INST] You are a helpful Python teacher.
Using the context below, explain in detail with examples like you are teaching a student.

Context:
                 MRCET 
54 
 
>>> 
 
Similarily we can also write,  
 
def f(arg): pass    # a function that does nothing (yet) 
 
class C: pass       # a class with no methods (yet) 
  
PYTHON PROGRAMMING                                      III YEAR/II SEM                  MRCET 
55 
 
UNIT – III 
FUNCTIONS, ARRAYS  
Fruitful functions: return values, parameters, local and global scope, function composition, 
recursion; Strings: string slices, immutability, string functions and methods, string module; 
Python arrays, Access the Elements of an Array, array methods. 
Functions, Arrays: 
Fruitful functions: 
We write functions that return values, which we will call  fruitful functions. We have seen 
the return statement before, but in a fruitful function the  return statement includes a  return 
value. This statement means: "Return immediately from